In [ ]:
#| default_exp edit_interactive

## Edit-interactive plan execution

Run a tiny Lisette agent loop against one notebook at a time. The inner agent receives the user plan plus a single compact notebook view, then can mutate only that notebook through scoped tools.

Sometimes a user wants an agent to carry out a bounded notebook edit rather than manually choose each `write_nb` or `update_cell` call. This notebook builds that inner edit loop: one notebook, one plan, a small set of notebook-aware tools, and a final diff.

The edit loop is for bounded delegation, not open-ended repository work. Its job is to give an inner agent a stable notebook view, a small tool belt, and revision-aware feedback so a single notebook can be edited and reviewed without exposing raw notebook JSON.

```python
execute_plan("nbs/02_write.ipynb", "Add an example after the write_nb docs", max_steps=4)
```

### Production contract

The edit-interactive loop is experimental. It stays out of the production core unless it has focused contract tests for bounded scope, stable notebook views, deterministic tool results, final diffs, and clear failure behavior when an inner edit cannot be completed.


In [ ]:
from contextlib import redirect_stdout
from io import StringIO
import nbskill.edit_interactive as ei
from fastcore.nbio import mk_cell, new_nb
from fastcore.nbio import read_nb as _read_tmp_nb, write_nb as _write_tmp_nb
from nbskill.edit_interactive import notebook_view as _example_notebook_view
from nbskill.foundation import demo_path, remove_demo_path, write_demo_notebook

In [ ]:
with write_demo_notebook("08_edit_interactive_example.ipynb") as path:
    _write_tmp_nb(new_nb([
        mk_cell("#| default_exp demo"),
        mk_cell("#| export\ndef answer():\n    return 42"),
    ]), path)
    print("\n".join(_example_notebook_view(path).splitlines()[:6]))

In [ ]:
#| export
import ast
import json
import os
import re
import traceback
from contextlib import redirect_stderr, redirect_stdout
from dataclasses import dataclass, field
from io import StringIO
from pathlib import Path
from threading import Lock
from fastcore.nbio import read_nb
from nbskill.edit import NotebookEditor
from nbskill.foundation import cap_text, cell_source, chapter_spans, commit_notebook, parse_one_cell
from nbskill.graph import symbol_usage_summary
from nbskill.knowledge import reference_query
from nbskill.parallel import notebook_locks
from nbskill.review import diff_nb

_CAPTURE_LOCK = Lock()

In [ ]:
#| export
def _save_notebook(nb, path):
    return commit_notebook(path, nb, before=read_nb(path))

### The inner-agent contract

The system prompt is intentionally narrow. The inner agent edits exactly one notebook, uses only the provided tools, prefers stable cell ids, and stops with a summary when the plan is done.

In [ ]:
#| export
EDIT_INTERACTIVE_SYSTEM = """You are an nbskill notebook-editing subagent.
You may edit one or more target notebooks in one repository. You receive the
project description, a focused knowledge summary, optional caller/callee impact
for the symbols in scope, and a managed notebook context prepared by a separate
context-management step. There is no project-scope lookup tool because that
scope is injected into this prompt.

Use only the provided editing tools: str_replace, edit_cell, add_cell,
delete_cell, execute_cell, and query_knowledge. The context-management tools are
not available in this edit step. When more than one notebook is in scope, pass
the notebook path/name to edit and execution tools so the target is explicit.
Keep the work scoped to one small specific task.

For new behavior, use this loop:
1. Experiment: write the smallest code that explores the idea, execute it, and
   inspect the result before treating it as correct.
2. Function: turn the working experiment into a focused function. Add a Markdown
   cell directly above the exported code explaining why the function is needed
   and why it is useful.
3. Example: add an example cell directly below the function showing how it works.
   If the example is slow or produces artifacts, add `#| eval: false`.
4. Test: add a focused test cell directly below the example so future changes
   keep the same output.

Use execute_cell normally to continue from the last executed cell through the
target cell, like a live notebook kernel. Pass rerun_all=True when earlier cells
changed and the whole notebook state must be rebuilt. Exceptions are returned to
you as tool output; inspect them, edit, and execute again. Stop as soon as the
requested notebook change is complete. Your final message must summarize what
changed, what was executed, and what could not be completed.
"""

CONTEXT_SESSION_SYSTEM = """You manage the context for an nbskill editing run.
You do not edit notebooks. Use only the context tools to open, hide, remove, or
summarize managed context messages so the next editing step sees the smallest
useful context. Prefer opening the few chapters directly relevant to the plan,
hiding full chapters after they are no longer useful, and leaving the index
visible. Stop once the context is prepared or cleaned.
"""

In [ ]:
#| export
def capture_call_text(func, **kwargs):
    "Run `func` and return captured stdout/stderr, or the return value."
    out, err = StringIO(), StringIO()
    with _CAPTURE_LOCK, redirect_stdout(out), redirect_stderr(err):
        result = func(**kwargs)
    chunks = []
    if out.getvalue(): chunks.append(out.getvalue().rstrip())
    if err.getvalue(): chunks.append(err.getvalue().rstrip())
    if result is not None and not chunks: chunks.append(str(result))
    return "\n".join(chunk for chunk in chunks if chunk)

### A stable notebook view

The edit loop needs a text representation that is compact enough for a model but precise enough for safe edits. `notebook_view` includes ids, cell types, and full cell source.

In [ ]:
#| export
def notebook_view(path, revision=0):
    "Render one notebook as a compact, stable text view."
    path = Path(path)
    with notebook_locks(path):
        nb = read_nb(path)
        lines = [f"Notebook: {path}", f"Revision: {revision}", ""]
        for idx, cell in enumerate(nb.cells):
            lines.append(f"CELL {idx} id={cell.id} type={cell.cell_type}")
            lines.append("<<<SOURCE")
            lines.append(cell_source(cell).rstrip())
            lines.append("SOURCE")
            lines.append("")
        return "\n".join(lines).rstrip() + "\n"


def _split_notebooks(notebooks):
    if notebooks is None: return []
    if isinstance(notebooks, (str, Path)): return [item.strip() for item in str(notebooks).split(",") if item.strip()]
    return [str(item) for item in notebooks if str(item).strip()]


def _target_paths(notebooks):
    paths = [Path(item) for item in _split_notebooks(notebooks)]
    if not paths: raise ValueError("At least one target notebook is required.")
    seen, duplicates = set(), []
    for path in paths:
        key = path.as_posix()
        if key in seen: duplicates.append(key)
        seen.add(key)
    if duplicates: raise ValueError(f"Duplicate notebook target(s): {', '.join(sorted(duplicates))}")
    return paths


def _target_slug(path):
    rel = Path(path).with_suffix("").as_posix()
    return re.sub(r"[^A-Za-z0-9_.-]+", "-", rel).strip("-.") or "notebook"


def _target_label(paths):
    paths = [Path(path) for path in paths]
    if len(paths) == 1: return str(paths[0])
    return ", ".join(str(path) for path in paths)


def _resolve_target_path(paths, notebook=None):
    paths = [Path(path) for path in paths]
    name = _none_if_blank(notebook)
    if name is None:
        if len(paths) == 1: return paths[0]
        raise ValueError("notebook is required when an edit session has multiple target notebooks")
    matches = []
    for path in paths:
        choices = {str(path), path.as_posix(), path.name, path.stem, _target_slug(path)}
        if str(name) in choices: matches.append(path)
    if len(matches) == 1: return matches[0]
    if not matches: raise ValueError(f"Unknown target notebook {name!r}; choose one of: {_target_label(paths)}")
    raise ValueError(f"Ambiguous target notebook {name!r}")


def notebooks_view(paths, revision=0):
    "Render one or more notebooks as compact stable text."
    return "\n".join(notebook_view(path, revision=revision).rstrip() for path in _target_paths(paths)) + "\n"

### Self-managed notebook context

A managed context keeps the same stable notebook addressing as `notebook_view`, but starts with only a chapter index. The inner agent can open, hide, remove, or edit tagged context messages by emitting compact text commands.

In [ ]:
#| export
class ManagedContextMessage:
    "One mutable context message controlled by a stable tag."
    def __init__(self, tag, purpose, content, visible=True, removed=False):
        self.tag = str(tag)
        self.purpose = str(purpose)
        self.content = str(content)
        self.visible = bool(visible)
        self.removed = bool(removed)


def _context_msg_get(msg, key):
    return msg.get(key) if isinstance(msg, dict) else getattr(msg, key)


def _context_msg_set(msg, key, value):
    if isinstance(msg, dict): msg[key] = value
    else: setattr(msg, key, value)


def _chapter_tag(cell, path=None):
    base = f"chapter:{getattr(cell, 'id', '')}"
    return base if path is None else f"notebook:{_target_slug(path)}:{base}"


def _public_symbols_in_source(source):
    try: tree = ast.parse(source)
    except SyntaxError: return []
    symbols = []
    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)) and not node.name.startswith("_"):
            symbols.append(node.name)
    return symbols


def _chapter_public_symbols(cells):
    symbols = []
    for cell in cells:
        if getattr(cell, "cell_type", None) == "code": symbols.extend(_public_symbols_in_source(cell_source(cell)))
    return symbols


def _first_markdown_paragraph(cells):
    for cell in cells:
        if getattr(cell, "cell_type", None) != "markdown": continue
        chunks, current = [], []
        for line in cell_source(cell).splitlines():
            stripped = line.strip()
            if stripped.startswith("#") or stripped.startswith("#|"): continue
            if stripped: current.append(stripped)
            elif current:
                chunks.append(" ".join(current))
                current = []
        if current: chunks.append(" ".join(current))
        if chunks: return cap_text(chunks[0], 360)
    return "No prose summary found."


def _chapter_summary(title, cells):
    symbols = _chapter_public_symbols(cells)
    parts = [f"{title}: {_first_markdown_paragraph(cells)}", f"cells={len(cells)}"]
    if symbols: parts.append("symbols=" + ", ".join(symbols[:8]))
    return " | ".join(parts)


def _chapter_source_view(path, span, cells):
    lines = [f"Full chapter context: {path}", f"Title: {span['title']}", f"Cells: {span['start']}:{span['end']}", ""]
    for idx in range(span["start"], span["end"]):
        cell = cells[idx]
        lines.append(f"CELL {idx} id={cell.id} type={cell.cell_type}")
        lines.append("<<<SOURCE")
        lines.append(cell_source(cell).rstrip())
        lines.append("SOURCE")
        lines.append("")
    return "\n".join(lines).rstrip() + "\n"


def managed_notebook_context(path, revision=0, previous=None):
    "Build mutable summary-first context messages for one or more target notebooks."
    paths = _target_paths(path)
    multi = len(paths) > 1
    old = {msg.tag: msg for msg in (previous or [])}
    index_lines = [
        "Managed notebook context",
        "Notebooks: " + _target_label(paths),
        f"Revision: {revision}",
        "",
        "Context is controlled by a separate context session. The edit session sees only visible messages.",
        "",
        "Chapter index:",
    ]
    messages = []
    for path in paths:
        with notebook_locks(path):
            nb = read_nb(path)
            spans = chapter_spans(nb.cells, levels=range(1, 7), fallback="Notebook")
            if multi:
                index_lines.append("")
                index_lines.append(f"Notebook: {path}")
            for span in spans:
                cells = list(nb.cells[span["start"]:span["end"]])
                tag = _chapter_tag(nb.cells[span["start"]], path if multi else None)
                summary = _chapter_summary(span["title"], cells)
                index_lines.append(f"- {tag} title={span['title']!r} cells={span['start']}:{span['end']} summary={summary}")
                messages.append(ManagedContextMessage(
                    tag=tag,
                    purpose=f"full chapter: {path} :: {span['title']}",
                    content=_chapter_source_view(path, span, nb.cells),
                    visible=False,
                ))
    index = ManagedContextMessage("notebook:index", "chapter summary index", "\n".join(index_lines).rstrip() + "\n")
    rebuilt = [index, *messages]
    for msg in rebuilt:
        prior = old.get(msg.tag)
        if prior is None: continue
        if msg.tag != "notebook:index":
            msg.visible, msg.removed = prior.visible, prior.removed
        if prior.purpose.startswith("edited "):
            msg.purpose, msg.content = prior.purpose, prior.content
    return rebuilt


def render_managed_context(messages):
    "Render currently visible managed context messages."
    visible = [msg for msg in messages if msg.visible and not msg.removed]
    hidden = [msg for msg in messages if not msg.visible and not msg.removed]
    removed = [msg for msg in messages if msg.removed]
    lines = ["Self-managed context messages", ""]
    for msg in visible:
        lines.append(f"<message tag={msg.tag!r} purpose={msg.purpose!r}>")
        lines.append(msg.content.rstrip())
        lines.append("</message>")
        lines.append("")
    if hidden:
        lines.append("Hidden messages:")
        lines.extend(f"- {msg.tag}: {msg.purpose}" for msg in hidden)
        lines.append("")
    if removed:
        lines.append("Removed messages:")
        lines.extend(f"- {msg.tag}: {msg.purpose}" for msg in removed)
        lines.append("")
    return "\n".join(lines).rstrip() + "\n"

### Session state

`EditSession` tracks the notebook path, revision, timeout, and operation logs. The revision count makes it clear which tool calls changed the notebook during a plan.

In [ ]:
#| export
@dataclass
class EditSession:
    "Mutable state for one edit-interactive notebook run."
    path: Path | str | list
    target_paths: list[Path] = field(default_factory=list)
    timeout: int = 30
    revision: int = 0
    agent_id: str | None = None
    log_path: Path | None = None
    log: list[str] = field(default_factory=list)
    tool_log: list[str] = field(default_factory=list)
    history: list[dict] = field(default_factory=list)
    messages: list[dict] = field(default_factory=list)
    managed_context: list[ManagedContextMessage] = field(default_factory=list)
    context_command_log: list[dict] = field(default_factory=list)
    live_ns: dict = field(default_factory=lambda: {"__name__": "__main__"})
    live_ns_by_path: dict = field(default_factory=dict)
    executed_until_idx: int = -1
    executed_until_by_path: dict = field(default_factory=dict)
    chat: object | None = None
    notebook_msg_idx: int | None = None
    context_msg_idx: int | None = None
    def __post_init__(self):
        paths = self.target_paths or _target_paths(self.path)
        self.target_paths = [Path(path) for path in paths]
        self.path = self.target_paths[0]
        if self.agent_id is None: self.agent_id = _agent_notebook_id(_target_label(self.target_paths))
        if self.log_path is None: self.log_path = Path("log") / f"agent-{self.agent_id}.log"
    def target_path(self, notebook=None):
        "Resolve a user-facing notebook selector against this session."
        return _resolve_target_path(self.target_paths, notebook)
    def _state_key(self, path):
        return Path(path).as_posix()
    def live_state(self, path):
        key = self._state_key(path)
        if key not in self.live_ns_by_path:
            self.live_ns_by_path[key] = {"__name__": "__main__"}
        return self.live_ns_by_path[key]
    def executed_until(self, path):
        return self.executed_until_by_path.get(self._state_key(path), -1)
    def set_executed_until(self, path, idx):
        self.executed_until_by_path[self._state_key(path)] = idx
    def reset_live_state(self, path=None):
        "Reset the live notebook execution namespace."
        if path is None:
            self.live_ns = {"__name__": "__main__"}
            self.live_ns_by_path = {}
            self.executed_until_idx = -1
            self.executed_until_by_path = {}
            return
        self.live_ns_by_path[self._state_key(path)] = {"__name__": "__main__"}
        self.executed_until_by_path[self._state_key(path)] = -1
    def refresh_view(self):
        "Replace the notebook context message in the active Lisette history."
        if self.chat is None: return
        if self.managed_context and self.context_msg_idx is not None:
            msg = self.chat.hist[self.context_msg_idx]
            self.managed_context = managed_notebook_context(self.target_paths, self.revision, previous=self.managed_context)
            _context_msg_set(msg, "content", render_managed_context(self.managed_context))
            return
        if self.notebook_msg_idx is None: return
        msg = self.chat.hist[self.notebook_msg_idx]
        _context_msg_set(msg, "content", notebooks_view(self.target_paths, self.revision))
    def record_message(self, role, content):
        "Append one agent message to memory and the run log."
        item = {"revision": self.revision, "role": role, "content": str(content)}
        self.messages.append(item)
        _append_agent_log(self, "message", item)
    def record(self, message):
        "Append an operation to the session log."
        self.log.append(f"r{self.revision}: {message}")
        _append_agent_log(self, "operation", {"revision": self.revision, "message": message})
    def record_tool(self, name, detail=""):
        "Append one tool use to the session history."
        suffix = f"({detail})" if detail else "()"
        self.tool_log.append(f"r{self.revision}: {name}{suffix}")
        item = {"revision": self.revision, "tool": name}
        if detail: item["detail"] = detail
        self.history.append(item)
        _append_agent_log(self, "tool", item)
    def record_context_command(self, command, status="ok", detail=""):
        "Append one managed-context command to the session history."
        item = {"revision": self.revision, "status": status, **command}
        if detail: item["detail"] = detail
        self.context_command_log.append(item)
        _append_agent_log(self, "context_command", item)

In [ ]:
#| export
_CTX_SIMPLE_RE = re.compile(r"\[\[ctx:(open|hide|remove)\s+([^\]]+)\]\]")
_CTX_EDIT_RE = re.compile(r"\[\[ctx:edit\s+([^\]]+)\]\](.*?)\[\[/ctx:edit\]\]", re.S)


def find_context_command(text):
    "Return the first complete managed-context command in `text`."
    simple = _CTX_SIMPLE_RE.search(text)
    edit = _CTX_EDIT_RE.search(text)
    matches = [match for match in [simple, edit] if match is not None]
    if not matches: return None
    match = min(matches, key=lambda item: item.start())
    if match.re is _CTX_EDIT_RE:
        action, tag, content = "edit", match.group(1).strip(), match.group(2)
    else:
        action, tag, content = match.group(1), match.group(2).strip(), None
    return {
        "action": action,
        "tag": tag,
        "content": content,
        "start": match.start(),
        "end": match.end(),
        "raw": match.group(0),
    }


def _managed_message_by_tag(messages, tag):
    matches = [msg for msg in messages if msg.tag == tag]
    if len(matches) == 1: return matches[0]
    if not matches: raise ValueError(f"No managed context message has tag {tag!r}")
    raise ValueError(f"Multiple managed context messages have tag {tag!r}")


def apply_context_command(session, command):
    "Apply one managed-context command to an edit session."
    command = dict(command)
    tag, action = command.get("tag", ""), command.get("action", "")
    msg = _managed_message_by_tag(session.managed_context, tag)
    if action == "open":
        msg.visible, msg.removed = True, False
    elif action == "hide":
        msg.visible = False
    elif action == "remove":
        msg.removed, msg.visible = True, False
    elif action == "edit":
        msg.content = str(command.get("content") or "")
        msg.purpose = "edited " + msg.purpose
        msg.visible, msg.removed = True, False
    else:
        raise ValueError(f"Unknown managed context action {action!r}")
    public = {key: command.get(key) for key in ("action", "tag")}
    session.record_context_command(public)
    session.refresh_view()
    return f"context {action} applied to {tag}"


def _stream_chunk_text(chunk):
    if isinstance(chunk, str): return chunk
    if isinstance(chunk, bytes): return chunk.decode("utf-8", errors="replace")
    try:
        content = chunk.choices[0].delta.content
        return "" if content is None else str(content)
    except (AttributeError, IndexError, TypeError):
        pass
    try:
        content = chunk.choices[0].message.content
        return "" if content is None else str(content)
    except (AttributeError, IndexError, TypeError):
        return ""


def _close_stream(stream):
    close = getattr(stream, "close", None)
    if callable(close): close()


def _chat_response_text(response):
    "Return assistant text from a Lisette response-like object."
    if isinstance(response, str): return response
    text = _stream_chunk_text(response)
    if text: return text
    return str(response or "")


def _chat_allows_nonstream(chat):
    "Return whether this model can recover with a non-streaming call."
    model = str(getattr(chat, "model", ""))
    return not model.startswith("chatgpt/")


def _chat_nonstream(session, prompt, max_steps):
    "Run one non-streaming continuation when provider streaming fails."
    if not _chat_allows_nonstream(session.chat):
        raise ValueError(f"{session.chat.model} requires stream=True; cannot use non-stream fallback")
    response = session.chat(prompt, max_steps=max_steps, stream=False)
    return _chat_response_text(response)


def run_chat_with_context_commands(session, prompt, max_steps=8, max_context_commands=8):
    "Run a streaming chat, intercepting managed-context text commands."
    if session.chat is None: raise ValueError("session.chat is required")
    output, next_prompt = [], prompt
    for _ in range(max_context_commands + 1):
        stream = None
        try:
            stream = session.chat(next_prompt, max_steps=max_steps, stream=True)
            buffer, command = "", None
            for chunk in stream:
                piece = _stream_chunk_text(chunk)
                if not piece: continue
                buffer += piece
                command = find_context_command(buffer)
                if command:
                    _close_stream(stream)
                    break
        except BaseException as exc:
            _close_stream(stream)
            if not session.context_command_log: raise
            detail = f"{type(exc).__name__}: {exc}"
            if not _chat_allows_nonstream(session.chat): raise
            session.record_context_command(
                {"action": "fallback", "tag": "managed-context"},
                status="ok",
                detail=detail,
            )
            output.append(_chat_nonstream(session, next_prompt, max_steps=max_steps))
            return "".join(output).strip()
        if command is None:
            output.append(buffer)
            return "".join(output).strip()
        output.append(buffer[:command["start"]])
        apply_context_command(session, command)
        next_prompt = (
            f"Managed context command applied: {command['action']} {command['tag']}. "
            "Continue the same task from just before the command. Do not repeat the command."
        )
    session.record_context_command({"action": "limit", "tag": "managed-context"}, status="error", detail="max_context_commands exceeded")
    return "".join(output).strip() or "managed context command limit reached"

In [ ]:
#| export
def _none_if_blank(value):
    if value is None: return None
    value = str(value)
    return None if value.strip().lower() in {"", "none", "null"} else value

In [ ]:
#| export
def _edit_find_cell_by_id(cells, cell_id):
    matches = [(idx, cell) for idx, cell in enumerate(cells) if getattr(cell, "id", None) == str(cell_id)]
    if len(matches) == 1: return matches[0]
    if not matches: raise ValueError(f"No cell has id {cell_id!r}")
    raise ValueError(f"Multiple cells have id {cell_id!r}")

In [ ]:
#| export
def _finish_write(session, path, message, result):
    session.revision += 1
    session.record(message)
    session.refresh_view()
    detail = str(result.get("text") or "").strip()
    chunks = [message, f"revision={session.revision}"]
    if detail: chunks.extend(["", detail])
    return "\n".join(chunks)

In [ ]:
#| export
def _agent_notebook_id(path):
    rel = Path(path).with_suffix("").as_posix()
    rel = re.sub(r"[^A-Za-z0-9_.-]+", "-", rel).strip("-.")
    return rel or "notebook"

In [ ]:
#| export
def _append_agent_log(session, kind, payload):
    path = Path(session.log_path)
    path.parent.mkdir(parents=True, exist_ok=True)
    record = {
        "kind": kind,
        "agent_id": session.agent_id,
        "notebook": str(session.path),
        "target_notebooks": [str(path) for path in session.target_paths],
        "payload": payload,
    }
    path.open("a", encoding="utf-8").write(json.dumps(record, ensure_ascii=False) + "\n")

In [ ]:
#| export
def _project_description(path):
    start = Path(path).resolve().parent
    for root in [start, *start.parents]:
        readme = root / "README.md"
        if readme.exists(): return cap_text(readme.read_text(encoding="utf-8"), 2400)
    return "No README.md project description was found."

In [ ]:
#| export
def _knowledge_context(query, top_k=3):
    try: result = reference_query(query, top_k=top_k, current_repo=".")
    except BaseException as exc: return f"Knowledge query unavailable: {type(exc).__name__}: {exc}"
    hits = []
    for hit in result.get("hits", []):
        label = ".".join(item for item in [hit.get("module"), hit.get("symbol")] if item)
        source = cap_text(hit.get("source") or hit.get("docstring") or "", 700)
        hits.append(f"- {label or hit.get('path')}: {source}")
    return "\n".join(hits) if hits else "No relevant knowledge hits."

In [ ]:
#| export
def _symbol_impact_context(symbols):
    names = _split_notebooks(symbols)
    if not names: return "No symbols requested for caller/callee impact."
    try: return symbol_usage_summary(".", names)
    except BaseException as exc: return f"Symbol impact unavailable: {type(exc).__name__}: {exc}"

In [ ]:
#| export
def _subagent_context(path, plan, symbols=None):
    return "\n\n".join([
        "Project description:\n" + _project_description(path),
        "Knowledge summary:\n" + _knowledge_context(plan),
        "Caller/callee impact:\n" + _symbol_impact_context(symbols),
    ])

### The editing tool belt

The inner loop only gets four notebook operations: add, edit, run through a cell, and remove. Each operation validates inputs, refreshes the notebook view, and records a diff-like message for the final report.

In [ ]:
#| export
def _inserted_cell_ids(result):
    inserted = []
    for diff in result.get("diffs", []):
        inserted.extend(diff.get("inserted_cell_ids", []))
    return inserted


def make_edit_tools(session):
    "Create notebook-scoped editing tools for one edit-interactive session."
    def str_replace(old_str: str, new_str: str, notebook: str | None = None) -> str:
        "Replace one exact string occurrence anywhere in a target notebook."
        path = session.target_path(notebook)
        with notebook_locks(path):
            nb = read_nb(path)
            matches = []
            for idx, cell in enumerate(nb.cells):
                source = cell_source(cell)
                if old_str in source: matches.append((idx, cell, source))
        session.record_tool("str_replace", f"notebook={path}, matches={len(matches)}")
        if len(matches) != 1: raise ValueError(f"old_str matched {len(matches)} cells in {path}; expected exactly 1")
        idx, cell, before = matches[0]
        after = before.replace(old_str, new_str, 1)
        result = NotebookEditor(path, auto_feedback=False).replace_cell(cell.id, after, cell_type=cell.cell_type)
        msg = f"Replaced text in {path} id={cell.id}"
        return _finish_write(session, path, msg, result)
    def edit_cell(id: str, old_str: str, new_str: str, notebook: str | None = None) -> str:
        "Replace one exact string occurrence inside one cell."
        path = session.target_path(notebook)
        with notebook_locks(path):
            nb = read_nb(path)
            idx, cell = _edit_find_cell_by_id(nb.cells, id)
            before = cell_source(cell)
            count = before.count(old_str)
        session.record_tool("edit_cell", f"notebook={path}, id={id!r}, matches={count}")
        if count != 1: raise ValueError(f"old_str matched {count} times in {path} id={id}; expected exactly 1")
        after = before.replace(old_str, new_str, 1)
        result = NotebookEditor(path, auto_feedback=False).replace_cell(id, after, cell_type=cell.cell_type)
        msg = f"Edited text in {path} id={id}"
        return _finish_write(session, path, msg, result)
    def add_cell(after_id: str | None = None, content: str = "", notebook: str | None = None) -> str:
        "Add one cell to a target notebook."
        path = session.target_path(notebook)
        anchor = _none_if_blank(after_id)
        new_cell = parse_one_cell(content, "code")
        if anchor is not None:
            with notebook_locks(path):
                _edit_find_cell_by_id(read_nb(path).cells, anchor)
        session.record_tool("add_cell", f"notebook={path}, after_id={anchor!r}")
        result = NotebookEditor(path, auto_feedback=False).insert(
            anchor, cell_source(new_cell), cell_type=new_cell.cell_type
        )
        inserted = _inserted_cell_ids(result)
        new_id = inserted[0] if inserted else ""
        where = f"after id={anchor}" if anchor is not None else "at end"
        msg = f"Added cell id={new_id} to {path} {where}"
        return _finish_write(session, path, msg, result)
    def delete_cell(id: str, notebook: str | None = None) -> str:
        "Delete one cell from a target notebook."
        path = session.target_path(notebook)
        with notebook_locks(path):
            _edit_find_cell_by_id(read_nb(path).cells, id)
        session.record_tool("delete_cell", f"notebook={path}, id={id!r}")
        result = NotebookEditor(path, auto_feedback=False).delete(id)
        msg = f"Deleted cell id={id} from {path}"
        return _finish_write(session, path, msg, result)
    def _exec_python(source, filename, ns):
        tree = ast.parse(source, filename=filename, mode="exec")
        if not tree.body: return None
        if not isinstance(tree.body[-1], ast.Expr):
            exec(compile(tree, filename, "exec"), ns)
            return None
        prefix = ast.Module(body=tree.body[:-1], type_ignores=[])
        expr = ast.Expression(tree.body[-1].value)
        ast.fix_missing_locations(prefix)
        ast.fix_missing_locations(expr)
        if prefix.body: exec(compile(prefix, filename, "exec"), ns)
        return eval(compile(expr, filename, "eval"), ns)
    def _eval_false(source):
        return any(re.match(r"#\|\s*eval:\s*false\b", line.strip(), re.I) for line in source.splitlines()[:5])
    def _execute_one(path, idx, cell):
        source = cell_source(cell)
        header = f"CELL {idx} id={cell.id} notebook={path}"
        if cell.cell_type != "code": return f"{header} status=skipped reason=non-code"
        if _eval_false(source): return f"{header} status=skipped reason=eval-false"
        out, err = StringIO(), StringIO()
        status, display, tb = "ok", None, ""
        filename = f"{path}::{cell.id}"
        ns = session.live_state(path)
        with _CAPTURE_LOCK, redirect_stdout(out), redirect_stderr(err):
            try: display = _exec_python(source, filename, ns)
            except BaseException:
                status = "error"
                tb = traceback.format_exc()
        chunks = [f"{header} status={status}"]
        if out.getvalue(): chunks.append("stdout:\n" + out.getvalue().rstrip())
        if err.getvalue(): chunks.append("stderr:\n" + err.getvalue().rstrip())
        if display is not None: chunks.append("display:\n" + repr(display))
        if tb: chunks.append("traceback:\n" + tb.rstrip())
        return "\n".join(chunks)
    def execute_cell(id: str, rerun_all: bool = False, notebook: str | None = None) -> str:
        "Execute through one cell in the live notebook state."
        path = session.target_path(notebook)
        with notebook_locks(path):
            nb = read_nb(path)
            target_idx, _ = _edit_find_cell_by_id(nb.cells, id)
            cells = list(nb.cells)
        if rerun_all: session.reset_live_state(path)
        start = session.executed_until(path) + 1
        if target_idx < start: start = target_idx
        session.record_tool("execute_cell", f"notebook={path}, id={id!r}, rerun_all={rerun_all}, start={start}, target={target_idx}")
        reports, status = [], "ok"
        for idx in range(start, target_idx + 1):
            report = _execute_one(path, idx, cells[idx])
            reports.append(report)
            session.set_executed_until(path, idx)
            if " status=error" in report:
                status = "error"
                break
        if not reports:
            reports.append(f"CELL {target_idx} id={id} notebook={path} status=skipped reason=already-executed")
        session.record(f"Executed {path} cells {start}:{target_idx} ({status})")
        return "\n\n".join([f"status={status}", *reports])
    def query_knowledge(query: str, top_k: int = 3) -> str:
        "Search the reference knowledge base."
        session.record_tool("query_knowledge", f"top_k={top_k}")
        return _knowledge_context(query, top_k=top_k)
    return [str_replace, edit_cell, add_cell, delete_cell, execute_cell, query_knowledge]

In [ ]:
#| export
def make_context_tools(session):
    "Create managed-context tools for a separate context session."
    def open_context(tag: str) -> str:
        "Show a hidden managed-context message."
        return apply_context_command(session, {"action": "open", "tag": tag})
    def hide_context(tag: str) -> str:
        "Hide a visible managed-context message without removing it."
        return apply_context_command(session, {"action": "hide", "tag": tag})
    def remove_context(tag: str) -> str:
        "Remove a managed-context message from later renders."
        return apply_context_command(session, {"action": "remove", "tag": tag})
    def edit_context(tag: str, content: str) -> str:
        "Replace one managed-context message with concise custom content."
        return apply_context_command(session, {"action": "edit", "tag": tag, "content": content})
    def context_view() -> str:
        "Return the currently visible managed context."
        session.record_context_command({"action": "view", "tag": "managed-context"})
        return render_managed_context(session.managed_context)
    return [open_context, hide_context, remove_context, edit_context, context_view]


def make_chat(model, tools, hist, system_prompt=EDIT_INTERACTIVE_SYSTEM):
    "Create the Lisette chat object for edit-interactive."
    from lisette import Chat
    return Chat(model, sp=system_prompt, tools=tools, hist=hist, stream=str(model).startswith("chatgpt/"))


def run_context_session(session, model, prompt, max_steps=4):
    "Run one separate context-management step and restore the active edit chat."
    if not session.managed_context: return ""
    prior_chat, prior_context_idx = session.chat, session.context_msg_idx
    hist = [{"role": "user", "content": render_managed_context(session.managed_context)}]
    chat = make_chat(model, tools=make_context_tools(session), hist=hist, system_prompt=CONTEXT_SESSION_SYSTEM)
    session.chat = chat
    session.context_msg_idx = len(chat.hist) - 1
    try:
        result = chat(prompt, max_steps=max_steps, return_all=True)
    except BaseException as exc:
        summary = f"context session failed: {type(exc).__name__}: {exc}"
    else:
        summary = response_text(result).strip() or "(no context response)"
    finally:
        session.chat, session.context_msg_idx = prior_chat, prior_context_idx
    session.record_message("context", summary)
    return summary

In [ ]:
#| export
def response_text(response):
    "Extract readable text from a Lisette response or response list."
    if isinstance(response, list) and response: response = response[-1]
    try:
        message = response.choices[0].message
        content = message.content
    except (AttributeError, IndexError, TypeError):
        return "" if response is None else str(response)
    if isinstance(content, list):
        return "".join(str(item.get("text", item)) if isinstance(item, dict) else str(item) for item in content)
    return "" if content is None else str(content)


def plan_result_text(result):
    "Return the human-readable text for an execute_plan result."
    if not isinstance(result, dict): return "" if result is None else str(result)
    if result.get("text"): return str(result["text"])
    sections = [
        "edit-interactive complete",
        "",
        "Final response:",
        str(result.get("summary") or "(no final response)"),
        "",
        "Tools used:",
        "\n".join(item.get("detail", item.get("tool", "")) for item in result.get("history", [])) or "(no tool calls)",
    ]
    return "\n".join(sections).rstrip()

In [ ]:
#| export
def final_diff(path):
    "Return a nbdev code-cell diff, or a clear unavailable message."
    try:
        with notebook_locks(path):
            return capture_call_text(diff_nb, path=str(path))
    except BaseException as exc:
        detail = str(exc)
        if "Could not find notebook" in detail or "No git repository found" in detail:
            return (
                "Code-cell diff against HEAD is unavailable because this notebook has no git baseline. "
                "This is expected for new or untracked notebooks."
            )
        return f"Code-cell diff unavailable: {type(exc).__name__}: {exc}"


def final_diffs(paths):
    "Return code-cell diffs for one or more notebooks."
    chunks = []
    for path in _target_paths(paths):
        chunks.extend([f"## {path}", final_diff(path).strip()])
    return "\n\n".join(chunks).rstrip()

### Running a plan

`execute_plan` packages the user's plan, the current notebook view, and the notebook tools into a Lisette chat. The result includes the final answer, tool log, operation log, revision, and code-cell diff.

In [ ]:
#| export
def execute_plan(
    notebook: str | list,
    plan: str,
    model: str | None = None,
    max_steps: int = 8,
    timeout: int = 30,
    dry_run: bool = False,
    symbols: str | None = None,
    injected_context: str | None = None,
    managed_context: bool = True,
    max_context_commands: int = 8,
    context_steps: int = 2,
) -> dict:
    "Execute `plan` against one or more notebooks using bounded subagent steps."
    paths = _target_paths(notebook)
    missing = [str(path) for path in paths if not path.exists()]
    if missing: raise ValueError(f"Notebook does not exist: {', '.join(missing)}")
    model = model or os.environ.get("NBSKILL_AGENT") or "chatgpt/gpt-5.4-mini"
    project_context = injected_context or _subagent_context(paths[0], plan, symbols=symbols)
    initial_messages = managed_notebook_context(paths, revision=0) if managed_context else []
    initial_view = render_managed_context(initial_messages) if managed_context else notebooks_view(paths, revision=0)
    if dry_run:
        view_label = "Initial managed notebook context:" if managed_context else "Initial notebook view:"
        text = "\n".join([
            "notebook subagent dry run", "", f"Notebooks: {_target_label(paths)}",
            f"Model: {model}", f"Max steps: {max_steps}", f"Managed context: {managed_context}",
            "Plan:", plan, "", "Injected project context:", project_context, "", view_label, initial_view,
        ]).rstrip()
        return {"summary": "Dry run only; no subagent was invoked and no notebook edits were made.", "history": [], "context_commands": [], "text": text}
    session = EditSession(path=paths, target_paths=paths, timeout=timeout, managed_context=initial_messages)
    if managed_context:
        prep = "Prepare context for the edit step. Open only chapters likely needed for this plan.\nPlan:\n" + plan
        run_context_session(session, model, prep, max_steps=context_steps)
        initial_view = render_managed_context(session.managed_context)
    hist = [
        {"role": "user", "content": "Injected project context:\n" + project_context},
        {"role": "user", "content": f"Plan:\n{plan}"},
        {"role": "user", "content": initial_view},
    ]
    for item in hist: session.record_message(item["role"], item["content"])
    chat = make_chat(model, tools=make_edit_tools(session), hist=hist)
    session.chat = chat
    if managed_context: session.context_msg_idx = len(chat.hist) - 1
    else: session.notebook_msg_idx = len(chat.hist) - 1
    session.refresh_view()
    prompt = (
        "Execute the plan with the experiment, function, example, and test loop from the system prompt. "
        "Use execute_cell to validate the experiment and the final example or test when practical. "
        "Context-management tools are not available in this edit step; use the visible context you were given. "
        "Stop when the notebook change is complete."
    )
    session.record_message("user", prompt)
    try:
        result = chat(prompt, max_steps=max_steps, return_all=True)
    except BaseException as exc:
        result = f"notebook subagent failed: {type(exc).__name__}: {exc}"
    if not isinstance(result, (list, str, bytes, dict)) and hasattr(result, "__next__"):
        result = list(result)
    summary = response_text(result).strip() or "(no final response)"
    session.record_message("assistant", summary)
    if managed_context:
        cleanup = "Clean up context after the edit step. Hide full chapters that are no longer useful and leave the index visible."
        run_context_session(session, model, cleanup, max_steps=context_steps)
    for path in paths:
        _save_notebook(read_nb(path), path)
    session.record("Exported notebook(s) after subagent run")
    diff = final_diffs(paths).strip()
    text = "\n".join([
        "notebook subagent complete", "", "Final response:", summary, "", "Tools used:",
        "\n".join(session.tool_log) if session.tool_log else "(no tool calls)", "", "Context commands:",
        "\n".join(f"r{item['revision']}: {item['action']} {item['tag']} ({item['status']})" for item in session.context_command_log) if session.context_command_log else "(no context commands)",
        "", "Operation log:", "\n".join(session.log) if session.log else "(no notebook operations)", "",
        f"Final revision: {session.revision}", f"Agent log: {session.log_path}", "", "Notebook diff:", diff,
    ]).rstrip()
    return {
        "summary": summary, "history": session.history, "context_commands": list(session.context_command_log),
        "messages": session.messages, "text": text, "model": model, "notebook": str(paths[0]),
        "notebooks": [str(path) for path in paths], "revision": session.revision, "operations": list(session.log),
        "log_path": str(session.log_path), "diff": diff,
    }

In [ ]:
#| export
# `_split_notebooks` is defined near the notebook rendering helpers because both context and execution use it.

In [ ]:
#| export
def execute_project_plan(
    plan: str,
    notebooks: str | None = None,
    model: str | None = None,
    max_steps: int = 8,
    timeout: int = 30,
    dry_run: bool = True,
    symbols: str | None = None,
) -> str:
    "Launch one edit-interactive session that can touch multiple notebooks."
    targets = _split_notebooks(notebooks)
    if not targets: raise ValueError("Pass one or more notebooks to execute_project_plan.")
    result = execute_plan(
        notebook=targets, plan=plan, model=model, max_steps=max_steps,
        timeout=timeout, dry_run=dry_run, symbols=symbols,
    )
    return "\n".join([
        "project subagent coordinator", "", f"Dry run: {dry_run}", f"Targets: {len(targets)}", "", plan_result_text(result),
    ]).rstrip()

In [ ]:
with write_demo_notebook("08_edit_view.ipynb") as path:
    _write_tmp_nb(new_nb([
        mk_cell("#| default_exp sample"),
        mk_cell("#| export\ndef public():\n    return 1"),
        mk_cell("assert public() == 1"),
    ]), path)
    view = ei.notebook_view(path, revision=3)
    assert "Revision: 3" in view
    assert "type=code" in view
    assert "type=code" in view
    assert "public()" in view

In [ ]:
with write_demo_notebook("08_edit_tools.ipynb") as path:
    first = mk_cell("#| default_exp sample")
    second = mk_cell("x = 1")
    _write_tmp_nb(new_nb([first, second]), path)
    session = ei.EditSession(path=path)
    class FakeChat:
        def __init__(self): self.hist = [{"role": "user", "content": "plan"}, {"role": "user", "content": ei.notebook_view(path)}]
    session.chat = FakeChat()
    session.notebook_msg_idx = 1
    str_replace, edit_cell, add_cell, delete_cell, execute_cell, query_knowledge = ei.make_edit_tools(session)
    replaced = str_replace("x = 1", "x = 10")
    assert "Replaced text" in replaced
    added = add_cell(second.id, "y = 2")
    nb = _read_tmp_nb(path)
    assert "Added cell" in added
    assert len(nb.cells) == 3
    assert "y = 2" in session.chat.hist[1]["content"]
    new_id = nb.cells[-1].id
    edited = edit_cell(new_id, "2", "3")
    assert "Edited text" in edited
    nb = _read_tmp_nb(path)
    assert nb.cells[-1].source == "y = 3"
    removed = delete_cell(new_id)
    assert "Deleted cell" in removed
    assert len(_read_tmp_nb(path).cells) == 2
    assert session.revision == 4
    assert session.log_path.exists()

In [ ]:
#| hide
with write_demo_notebook("08_edit_managed_context.ipynb") as path:
    intro = mk_cell("## Intro\nThis chapter explains the public answer.", cell_type="markdown")
    body = mk_cell("#| export\ndef public_answer():\n    return 42")
    details = mk_cell("## Details\nExtra notes live here.", cell_type="markdown")
    _write_tmp_nb(new_nb([intro, body, details]), path)
    session = ei.EditSession(path=path)
    session.managed_context = ei.managed_notebook_context(path)
    class FakeContextChat:
        def __init__(self): self.hist = [{"role": "user", "content": ""}]
    session.chat = FakeContextChat()
    session.context_msg_idx = 0
    rendered = ei.render_managed_context(session.managed_context)
    tag = f"chapter:{intro.id}"
    assert tag in rendered
    assert "public_answer" in rendered
    assert "return 42" not in rendered

    ei.apply_context_command(session, {"action": "open", "tag": tag})
    assert "return 42" in session.chat.hist[0]["content"]
    ei.apply_context_command(session, {"action": "hide", "tag": tag})
    assert "return 42" not in session.chat.hist[0]["content"]
    ei.apply_context_command(session, {"action": "open", "tag": tag})
    add_cell = ei.make_edit_tools(session)[2]
    add_cell(body.id, "added_value = 3")
    assert "added_value = 3" in session.chat.hist[0]["content"]
    ei.apply_context_command(session, {"action": "edit", "tag": tag, "content": "custom context"})
    assert "custom context" in session.chat.hist[0]["content"]
    ei.apply_context_command(session, {"action": "remove", "tag": tag})
    assert "custom context" not in session.chat.hist[0]["content"]

In [ ]:
with write_demo_notebook("08_edit_update.ipynb") as path:
    cell = mk_cell("x = 1")
    _write_tmp_nb(new_nb([cell]), path)
    session = ei.EditSession(path=path)
    edit_cell = ei.make_edit_tools(session)[1]
    result = edit_cell(cell.id, "1", "2")
    assert f"Edited text in {path} id={cell.id}" in result
    assert read_nb(path).cells[0].source == "x = 2"

In [ ]:
with write_demo_notebook("08_edit_missing.ipynb") as path:
    _write_tmp_nb(new_nb([mk_cell("x = 1")]), path)
    session = ei.EditSession(path=path)
    delete_cell = ei.make_edit_tools(session)[3]
    try:
        delete_cell("missing")
    except ValueError as exc:
        assert "No cell has id" in str(exc)
    else:
        raise AssertionError("expected missing cell failure")

In [ ]:
with write_demo_notebook("08_edit_duplicate.ipynb") as path:
    _write_tmp_nb(new_nb([mk_cell("x = 1"), mk_cell("x = 1")]), path)
    session = ei.EditSession(path=path)
    str_replace = ei.make_edit_tools(session)[0]
    try:
        str_replace("x = 1", "x = 2")
    except ValueError as exc:
        assert "matched 2 cells" in str(exc)
    else:
        raise AssertionError("expected ambiguous replace failure")

In [ ]:
#| hide
from contextlib import contextmanager


@contextmanager
def _patched_make_chat(fake):
    old_make_chat = ei.make_chat
    ei.make_chat = fake
    try: yield
    finally: ei.make_chat = old_make_chat


class FakeChat:
    calls = []
    def __init__(self, model, sp, tools, hist):
        self.model, self.sp, self.tools, self.hist = model, sp, tools, hist
        FakeChat.calls.append({"kind": "init", "tools": [tool.__name__ for tool in tools], "system": sp})
    def __call__(self, msg, max_steps=20, return_all=False, stream=False):
        names = [tool.__name__ for tool in self.tools]
        FakeChat.calls.append({"kind": "call", "tools": names, "msg": msg, "stream": stream})
        if "open_context" in names:
            if "Prepare context" in msg: self.tools[0](stream_tag)
            elif "Clean up context" in msg: self.tools[1](stream_tag)
            return "context step done"
        assert "open_context" not in names
        add_cell = self.tools[2]
        execute_cell = self.tools[4]
        add_cell(None, "answer = 42")
        new_id = read_nb(path).cells[-1].id
        report = execute_cell(new_id)
        assert "status=ok" in report
        return "edit step done"


def fake_make_chat(model, tools, hist, system_prompt=ei.EDIT_INTERACTIVE_SYSTEM):
    return FakeChat(model, system_prompt, tools, hist)


with write_demo_notebook("08_edit_plan.ipynb") as path:
    absolute_path = path.resolve()
    first = mk_cell("## Demo\nA tiny managed chapter.", cell_type="markdown")
    _write_tmp_nb(new_nb([first]), absolute_path)
    stream_tag = f"chapter:{first.id}"
    agent_args = dict(model="fake", max_steps=2, timeout=2, injected_context="test context")
    FakeChat.calls = []
    with _patched_make_chat(fake_make_chat): result = ei.execute_plan(str(absolute_path), "Inspect the demo chapter.", **agent_args)
    text = ei.plan_result_text(result)

assert result["summary"] == "edit step done"
assert result["context_commands"][0]["action"] == "open"
assert result["context_commands"][0]["tag"] == stream_tag
assert any(call["kind"] == "init" and "open_context" in call["tools"] for call in FakeChat.calls)
assert any(call["kind"] == "init" and "add_cell" in call["tools"] and "open_context" not in call["tools"] for call in FakeChat.calls)
assert "Final response:" in text
assert "Tools used:" in text
assert "Agent log:" in text
assert "edit step done" in text

with write_demo_notebook("08_edit_plan_legacy.ipynb") as path:
    absolute_path = path.resolve()
    _write_tmp_nb(new_nb([mk_cell("#| default_exp sample")]), absolute_path)
    agent_args = dict(model="fake", max_steps=2, timeout=2, injected_context="test context", managed_context=False)
    FakeChat.calls = []
    with _patched_make_chat(fake_make_chat): result = ei.execute_plan(str(absolute_path), "Add an answer cell.", **agent_args)

assert result["summary"] == "edit step done"
assert [item["tool"] for item in result["history"]] == ["add_cell", "execute_cell"]
assert result["revision"] == 1
assert result["log_path"].endswith(".log")

with write_demo_notebook("08_edit_multi_a.ipynb") as path_a:
    with write_demo_notebook("08_edit_multi_b.ipynb") as path_b:
        a = mk_cell("# A\nFirst", cell_type="markdown")
        b = mk_cell("# B\nSecond", cell_type="markdown")
        _write_tmp_nb(new_nb([a, mk_cell("x = 1")]), path_a)
        _write_tmp_nb(new_nb([b, mk_cell("y = 2")]), path_b)
        messages = ei.managed_notebook_context([path_a, path_b])
        tags = [msg.tag for msg in messages if msg.tag != "notebook:index"]
        assert all(tag.startswith("notebook:") for tag in tags)
        session = ei.EditSession(path=[path_a, path_b], managed_context=messages)
        str_replace = ei.make_edit_tools(session)[0]
        try:
            str_replace("x = 1", "x = 3")
        except ValueError as exc:
            assert "notebook is required" in str(exc)
        else:
            raise AssertionError("expected explicit notebook target")
        str_replace("x = 1", "x = 3", notebook=path_a.name)
        assert "x = 3" in ei.notebook_view(path_a)
        assert "y = 2" in ei.notebook_view(path_b)